# Lecture 5.2 — Handoffs: Conversation Delegation to Specialist Agents

**Section 05 — Multi-Agent Orchestration & Guardrails**

In this notebook you will build multi-agent systems where a triage agent hands off full control of the conversation to a specialist agent, using every option the `handoff()` function exposes.

## Cell 1 — Install the OpenAI Agents SDK

This notebook uses the **OpenAI Agents SDK** (`openai-agents` on PyPI), the Python framework this course is built on.

The cell below installs a pinned version of the package so the examples in this notebook behave exactly as shown, regardless of when you run it. The `-q` flag keeps the install output quiet.

If the package is already present in your current Colab session (for example, you ran this cell once already), pip will detect that the version matches and skip the download, so it is safe to run this cell more than once.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00


## Cell 2 — Configure Your OpenAI API Key

Every agent in this notebook calls the OpenAI API, so you need an API key available in the environment before any agent runs.

This course uses **Colab Secrets** exclusively to store the key. Secrets are encrypted, scoped to your Google account, and never appear in the notebook file itself, which makes them the safest way to hold credentials in a shared or downloadable notebook.

**Steps to add your key in Colab:**

| Step | Action |
|---|---|
| 1 | Click the key icon (🔑) in the left sidebar of Colab to open the **Secrets** panel |
| 2 | Click **Add new secret** |
| 3 | Set the name to `OPENAI_API_KEY` |
| 4 | Paste your API key as the value |
| 5 | Toggle **Notebook access** on for this notebook |

Once the secret is added, the cell below reads it with `userdata.get()` and writes it into `os.environ`, which is where the OpenAI client library looks for it automatically.

**Running locally instead of Colab?** Skip the `userdata` call and set the environment variable in your terminal before starting Jupyter, for example `export OPENAI_API_KEY="sk-..."` on macOS/Linux or `setx OPENAI_API_KEY "sk-..."` on Windows.

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Set the Model Name

Every `Agent` in this notebook is built with a `model=MODEL_NAME` argument rather than a hardcoded model string. That means changing this one variable updates the model used by **every agent in the notebook** at once, which is useful when a newer model becomes available or you want to compare quality and cost.

The comment above the assignment links to OpenAI's models page, since the list of available models changes over time and this notebook cannot stay current on its own.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

| Import | Purpose |
|---|---|
| `datetime` | Timestamps the `on_handoff` callback log line later in the notebook |
| `dataclass` | Builds the lightweight `UserSession` context object used with `is_enabled` |
| `BaseModel` (pydantic) | Defines `EscalationData`, the schema the model fills in for a handoff with `input_type` |
| `Reasoning` | GPT‑5‑family reasoning control. Note this comes from `openai.types.shared`, **not** from `agents` — a common mix-up |
| `Agent` | Defines each agent (triage and specialist) |
| `HandoffCallItem` | Item type that appears in `result.new_items` when the model calls a handoff tool |
| `HandoffOutputItem` | Item type that appears once a handoff has completed and control has switched agents |
| `ItemHelpers` | Utility class with helpers such as `text_message_output()` for reading message content out of run items |
| `MessageOutputItem` | Item type for a plain assistant message, used here to show which agent produced it |
| `ModelSettings` | Configures per-agent generation settings such as `reasoning` and `verbosity` |
| `RunContextWrapper` | Wraps the context object passed into `Runner.run()`, given to `on_handoff` callbacks and `is_enabled` checks |
| `Runner` | Executes an agent (or chain of agents, via handoffs) against an input |
| `handoff` | Factory function that builds a customized `Handoff` object for use inside an `Agent`'s `handoffs=[]` list |

All of the `agents`-namespaced imports below come from the top-level `agents` package, matching how you have imported them in previous lectures.

In [4]:
import datetime
from dataclasses import dataclass

from pydantic import BaseModel
from openai.types.shared import Reasoning

from agents import (
    Agent,
    HandoffCallItem,
    HandoffOutputItem,
    ItemHelpers,
    MessageOutputItem,
    ModelSettings,
    RunContextWrapper,
    Runner,
    handoff,
)

## Cell 5 — How Handoffs Work

Before writing any handoff code, it helps to understand what actually happens under the hood when one agent hands off to another.

**Handoffs are exposed to the model as tools.** If a triage agent has a handoff to an agent named `Refund Agent`, that handoff shows up in the triage agent's tool list as a tool called `transfer_to_refund_agent`. The model does not know "handoff" as a special concept; from its point of view, calling the handoff tool is no different from calling any other tool.

**What happens when the model calls that tool:**

| Step | What the SDK does |
|---|---|
| 1 | If an `on_handoff` callback was registered, the SDK runs it |
| 2 | If an `input_filter` was registered, the SDK applies it to the conversation history (covered in Lecture 5.3) |
| 3 | The SDK switches the *active agent* to the target agent |

**The new agent receives the full conversation history by default.** This is the fundamental difference between a handoff and `as_tool()`, which you saw in an earlier lecture. With `as_tool()`, one agent calls another as a subroutine and gets a return value back. With a handoff, control genuinely transfers: the new agent takes over the conversation and produces the final output itself. The original agent does not see what happens next.

**How to tell what happened after a run:**

- `result.last_agent` tells you which agent was active when the run finished.
- `result.new_items` contains a `HandoffCallItem` (the model's decision to hand off) and a `HandoffOutputItem` (the completed handoff, recording which agent handed off to which) for every handoff that occurred.

**If the model requests more than one handoff in a single turn, only the first one executes.** Every additional handoff call in that turn receives a tool output telling it "Multiple handoffs detected, ignoring this one." This matters because it means your agent instructions should guide the model toward requesting one handoff at a time.

**Two ways to add a handoff to an agent's `handoffs=[]` list:**

1. Pass an `Agent` instance directly. This is the simplest form, and is what you will build first.
2. Pass a `Handoff` object built with the `handoff()` factory function. This unlocks custom tool names, custom descriptions, callbacks, structured input, and dynamic enable/disable, all of which you will build later in this notebook.

## Cell 6 — Simplest Form: Passing Agent Instances Directly

The simplest way to wire up a handoff is to pass `Agent` instances straight into another agent's `handoffs=[]` parameter. No `handoff()` call is required at all.

This cell builds three agents:

| Agent | Role |
|---|---|
| `math_agent` | Specialist for maths questions |
| `writing_agent` | Specialist for creative writing questions |
| `triage_agent` | Routes the request to whichever specialist fits, via `handoffs=[math_agent, writing_agent]` |

Each specialist uses `reasoning=Reasoning(effort="none")` and `verbosity="low"`, keeping responses fast and inexpensive for a course demo. `MODEL_NAME` is passed to every agent rather than a hardcoded string, consistent with Cell 3.

When `triage_agent` decides the request is a maths question, the SDK auto-generates a tool called `transfer_to_math_agent` for it (derived from the agent's `name`, converted to snake_case). Calling that tool is what triggers the handoff.

Run the cell and watch `result.last_agent.name`. It will report `"Math Agent"`, not `"Triage Agent"`, confirming that control genuinely transferred to the specialist rather than the triage agent summarizing the specialist's work itself.

In [5]:
math_agent = Agent(
    name="Math Agent",
    instructions=(
        "You are a mathematics specialist. "
        "Solve maths problems step by step."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

writing_agent = Agent(
    name="Writing Agent",
    instructions=(
        "You are a creative writing specialist. "
        "Help with essays, stories, and creative content."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are a triage agent. "
        "Route requests to the correct specialist. "
        "For maths questions use the Math Agent. "
        "For writing questions use the Writing Agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[math_agent, writing_agent],
)

result = await Runner.run(
    triage_agent,
    "What is the derivative of x squared?",
)

print("Final output:", result.final_output)
print("Last agent:", result.last_agent.name)

Final output: The derivative of \(x^2\) is:

\[
\frac{d}{dx}(x^2)=2x
\]

Step-by-step using the power rule:
- The power rule says \(\frac{d}{dx}(x^n)=n x^{n-1}\)
- Here \(n=2\)
- So \(\frac{d}{dx}(x^2)=2x^{1}=2x\)
Last agent: Math Agent


## Cell 7 — Reading the Handoff Trail in `new_items`

`result.last_agent` tells you the final destination, but `result.new_items` tells you the full story of how the run got there. This cell reruns the same `triage_agent` from Cell 6 with a writing question, then walks the item list and prints what each item represents.

Two item types matter most for understanding handoffs:

| Item type | What it tells you |
|---|---|
| `HandoffCallItem` | The model decided to call a handoff tool. `.agent` is the agent that made the call (the *source* agent) |
| `HandoffOutputItem` | The handoff finished executing. `.source_agent` is who handed off, `.target_agent` is who received control |

The loop below also checks for `MessageOutputItem`, since `.agent` on a message tells you which agent actually produced that piece of text. That is useful once you have several agents contributing to one run and want to attribute each message correctly, rather than assuming every message came from the agent you originally called.

Run the cell and read the printed item list top to bottom. You are looking at the complete routing decision log for this run: the moment the triage agent decided to hand off, and the moment the writing agent took over.

In [6]:
result = await Runner.run(
    triage_agent,
    "Help me write a short poem about autumn.",
)

print("Final output:", result.final_output)
print("Last agent:", result.last_agent.name)
print(f"\nTotal items: {len(result.new_items)}")

for i, item in enumerate(result.new_items):
    print(
        f"  Item {i}: {type(item).__name__} "
        f"(type={item.type})"
    )
    if isinstance(item, HandoffCallItem):
        print(f"    Called by: {item.agent.name}")
    elif isinstance(item, HandoffOutputItem):
        print(
            f"    Route: {item.source_agent.name} -> "
            f"{item.target_agent.name}"
        )
    elif isinstance(item, MessageOutputItem):
        text = ItemHelpers.text_message_output(item)
        print(
            f"    [{item.agent.name}]: {text[:60]}"
        )

Final output: Autumn drifts in softly,  
gold leaves touch the ground,  
the air grows crisp and quiet,  
while amber light hangs round.  

The trees let go so gently,  
their branches bare and wise,  
and sunset paints the evening  
in fire across the skies.
Last agent: Writing Agent

Total items: 3
  Item 0: HandoffCallItem (type=handoff_call_item)
    Called by: Triage Agent
  Item 1: HandoffOutputItem (type=handoff_output_item)
    Route: Triage Agent -> Writing Agent
  Item 2: MessageOutputItem (type=message_output_item)
    [Writing Agent]: Autumn drifts in softly,  
gold leaves touch the ground,  
t


## Cell 8 — Overriding the Tool Name and Description with `handoff()`

Passing an `Agent` instance directly (Cell 6) is convenient, but it leaves you with an auto-generated tool name and a generic description. When the model has to choose between several specialists, a vague description makes that choice less reliable.

The `handoff()` function lets you override both:

| Parameter | Default | What it controls |
|---|---|---|
| `tool_name_override` | `transfer_to_<agent_name>` | The literal tool name shown to the model |
| `tool_description_override` | `"Handoff to the {agent.name} agent to handle the request. {handoff_description}"` | The tool description the model reads to decide when to call it |

This cell builds a `science_agent` alongside the `math_agent` from Cell 6, then creates a new triage agent, `triage_custom`, whose `handoffs=[]` list uses `handoff()` for both destinations with custom names and descriptions. Precise descriptions like these are exactly what you would tune in a production system to reduce misrouting.

Note that `handoff()` always transfers to the one specific agent you pass as its first argument. It is not a router itself. If you need the model to choose between multiple destinations, you register one `handoff()` per destination and let the model pick the tool, exactly as this cell does with `route_to_maths` and `route_to_science`.

In [7]:
science_agent = Agent(
    name="Science Agent",
    instructions="You are a science specialist.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

triage_custom = Agent(
    name="Custom Triage Agent",
    instructions=(
        "You are a triage agent. "
        "Route maths to the maths specialist. "
        "Route science to the science specialist."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[
        handoff(
            math_agent,
            tool_name_override="route_to_maths",
            tool_description_override=(
                "Route this question to the maths specialist "
                "for arithmetic, algebra, or calculus."
            ),
        ),
        handoff(
            science_agent,
            tool_name_override="route_to_science",
            tool_description_override=(
                "Route this question to the science specialist "
                "for physics, chemistry, or biology."
            ),
        ),
    ],
)

result = await Runner.run(
    triage_custom,
    "What is Newton's second law?",
)

print("Final output:", result.final_output)
print("Last agent:", result.last_agent.name)

Final output: Newton’s second law states:

**Force = mass × acceleration**  
\[
F = ma
\]

So, the net force on an object equals its mass times its acceleration.
Last agent: Custom Triage Agent


## Cell 9 — `on_handoff`: Running Code the Moment a Handoff Fires

`on_handoff` is a callback the SDK invokes at the exact moment the handoff tool is called, before the target agent produces any output. It is the right place for side effects: logging, analytics, database writes, or kicking off a background fetch you know you will need.

When `input_type` is **not** set (as in this cell), the callback must accept exactly one parameter: the `RunContextWrapper`. The SDK checks this at handoff-build time and raises `UserError` if the parameter count is wrong, so a mismatched signature fails fast rather than at run time.

This cell defines `log_handoff`, a plain synchronous function that prints a timestamped log line, and attaches it to a handoff into `billing_agent` using `handoff(billing_agent, on_handoff=log_handoff)`. Both sync and async callbacks are supported; the SDK awaits the result if the function is a coroutine.

One thing `on_handoff` **cannot** do: choose a different destination. `handoff()` always transfers to the specific agent captured when it was created, in this case `billing_agent`, regardless of what the callback does. Use `on_handoff` for bookkeeping, not routing.

Run the cell and look for the `[HANDOFF]` log line in the output, printed before the billing agent's response. That ordering is the callback firing at handoff time, exactly as described above.

In [8]:
def log_handoff(ctx: RunContextWrapper) -> None:
    print(
        f"[HANDOFF] "
        f"{datetime.datetime.now().strftime('%H:%M:%S')} "
        f"Routing to billing specialist"
    )

billing_agent = Agent(
    name="Billing Agent",
    instructions="You are a billing and payment specialist.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

triage_with_callback = Agent(
    name="Triage with Callback",
    instructions=(
        "You are a triage agent. "
        "Route billing questions to the Billing Agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[
        handoff(
            billing_agent,
            on_handoff=log_handoff,
        ),
    ],
)

result = await Runner.run(
    triage_with_callback,
    "I need help with my invoice.",
)

print("Final output:", result.final_output)

[HANDOFF] 03:01:57 Routing to billing specialist
Final output: I can help with that. What’s the issue with your invoice—wrong amount, missing payment, duplicate charge, or something else?


## Cell 10 — `input_type`: Letting the Model Attach Structured Metadata to a Handoff

Cell 9's callback ran with no extra information beyond the context. Sometimes you want the *model itself* to supply a small piece of structured data at the moment it calls the handoff, for example why it is escalating and how urgent the issue is.

That is what `input_type` is for. You define a Pydantic model describing the payload, in this case `EscalationData` with a `reason` and a `priority` field, and pass it as `input_type=EscalationData` alongside `on_handoff=on_escalation`. Two things change compared to Cell 9:

| Aspect | Without `input_type` (Cell 9) | With `input_type` (this cell) |
|---|---|---|
| `on_handoff` signature | 1 parameter: `ctx` | 2 parameters: `ctx`, `input_data` |
| Where the payload comes from | Nowhere, callback has no extra data | The model generates JSON matching `EscalationData`, the SDK validates it, then passes the parsed object to `on_handoff` |

**`input_type` describes the handoff tool's call arguments only.** It is easy to confuse this with two other things it is *not*:

- It is **not** the next agent's main input. The escalation agent still receives the full conversation, exactly as in every other handoff in this notebook.
- It is **not** a replacement for `RunContextWrapper.context`. Context is application state you already have locally (user ID, session data, feature flags). `input_type` is metadata the *model* decides at the moment it hands off.

Use `input_type` for small, model-generated details: a reason, a detected language, a priority level, a short summary. Run the cell and check the `[ESCALATION]` log line for the reason and priority the model chose on its own.

In [9]:
class EscalationData(BaseModel):
    reason: str
    priority: str

async def on_escalation(
    ctx: RunContextWrapper,
    input_data: EscalationData,
) -> None:
    print(
        f"[ESCALATION] Reason: {input_data.reason} | "
        f"Priority: {input_data.priority}"
    )

escalation_agent = Agent(
    name="Escalation Agent",
    instructions=(
        "You are an escalation specialist for urgent issues."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

support_triage = Agent(
    name="Support Triage",
    instructions=(
        "You are a customer support triage agent. "
        "If the issue is urgent and needs escalation, "
        "use the escalation handoff and provide a reason "
        "and priority level."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[
        handoff(
            escalation_agent,
            on_handoff=on_escalation,
            input_type=EscalationData,
        ),
    ],
)

result = await Runner.run(
    support_triage,
    "This is urgent, my account has been locked and I "
    "have a presentation in 10 minutes!",
)

print("Final output:", result.final_output)

[ESCALATION] Reason: Customer reports account is locked and they have a presentation in 10 minutes, requiring immediate access and urgent assistance. | Priority: high
Final output: I’ve escalated this for urgent help right now.


## Cell 11 — `is_enabled`: Hiding a Handoff Dynamically

Every handoff so far has always been visible to the model. `is_enabled` lets you hide a handoff based on runtime state, the same pattern you already used for `@function_tool(is_enabled=...)` on ordinary tools earlier in the course.

`is_enabled` accepts either a plain `bool`, or a callable with the signature `(ctx, agent) -> bool` (sync or async). This cell uses the callable form. `premium_only` reads `ctx.context.is_premium` off a small `UserSession` dataclass and returns whether the current user qualifies for the premium handoff.

**When `is_enabled` returns `False`, the handoff is not just declined, it is invisible.** The model never sees `transfer_to_premium_agent` in its tool list at all, so it cannot attempt to call it, and there is nothing to reject.

The cell runs the same question twice against `dynamic_triage`, once with `UserSession(is_premium=False)` and once with `UserSession(is_premium=True)`, passed through `Runner.run(..., context=...)`. Compare the two outputs: for the free user, `Premium Agent` is not an option the triage agent can reach; for the premium user, the handoff is available and `last_agent.name` reports `"Premium Agent"`.

In [10]:
@dataclass
class UserSession:
    is_premium: bool

def premium_only(
    ctx: RunContextWrapper[UserSession],
    agent: Agent,
) -> bool:
    return ctx.context.is_premium

premium_agent = Agent(
    name="Premium Agent",
    instructions="You are a premium-tier specialist.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

dynamic_triage = Agent(
    name="Dynamic Triage",
    instructions=(
        "You are a triage agent. "
        "Route premium requests to the Premium Agent "
        "if available."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[
        handoff(
            premium_agent,
            is_enabled=premium_only,
        ),
    ],
)

# Free user - handoff hidden from model
free_result = await Runner.run(
    dynamic_triage,
    "I need premium support.",
    context=UserSession(is_premium=False),
)
print("Free user:", free_result.final_output)
print("Last agent:", free_result.last_agent.name)

# Premium user - handoff visible to model
premium_result = await Runner.run(
    dynamic_triage,
    "I need premium support.",
    context=UserSession(is_premium=True),
)
print("Premium user:", premium_result.final_output)
print("Last agent:", premium_result.last_agent.name)

Free user: Routing you to Premium Agent now.
Last agent: Dynamic Triage
Premium user: You’re now connected to premium support.
Last agent: Premium Agent


## Cell 12 — `handoff()` Parameter Reference

Now that you have used every option hands-on, here is the complete reference for the `handoff()` function in one place.

| Parameter | Default | Description |
|---|---|---|
| `agent` | required | The agent to hand off to. `handoff()` always transfers to this specific agent |
| `tool_name_override` | `transfer_to_<agent_name>` | Custom tool name shown to the model |
| `tool_description_override` | auto-generated description | Custom tool description shown to the model |
| `on_handoff` | `None` | Callback fired when the handoff is invoked. Takes 1 parameter (`ctx`) if `input_type` is unset, or 2 parameters (`ctx`, `input_data`) if it is set. Wrong parameter count raises `UserError` |
| `input_type` | `None` | A Pydantic model describing the handoff tool's call arguments. The model's generated JSON is validated against it before being passed to `on_handoff` |
| `input_filter` | `None` | Filters the conversation history passed to the next agent. Covered in Lecture 5.3 |
| `is_enabled` | `True` | A `bool`, or a callable `(ctx, agent) -> bool` (sync or async), controlling whether the model can see this handoff at all |
| `nest_handoff_history` | `None` | Per-handoff override of the run-level `nest_handoff_history` setting. Falls back to the run configuration when `None` |

**Two formulas worth remembering:**

- Default tool name: `transfer_to_<agent_name_snake_case>`
- Default tool description: `"Handoff to the {agent.name} agent to handle the request. {agent.handoff_description or ''}"`

This notebook did not cover `input_filter`, `nest_handoff_history`, or guardrails on handoffs. Those are next.